# Flight Price Prediction — Gradient Boosting

## How gradient boosting works (viva notes)

Gradient boosting is gradient descent where each step adds a new decision tree instead of updating parameters.

**Algorithm (for MSE loss):**

1. Start: predict the mean price for all rows. Compute residuals.
2. Fit a **shallow tree** (depth 3–5) to predict the residuals — not the prices, the errors.
3. Add that tree to the ensemble, scaled by `learning_rate`. Residuals shrink.
4. Fit another shallow tree to the **new residuals**. Add it in.
5. Repeat for `n_estimators` rounds.

The residuals at each step equal the **negative gradient of MSE** w.r.t. the current predictions — so this is literally gradient descent in function space.

**Key hyperparameters:**
- `n_estimators`: number of trees (boosting rounds). More = lower training error but risk of overfitting.
- `learning_rate`: shrinks each tree's contribution. Lower rate → need more trees, but generalises better. Typical range: 0.05–0.2.
- `max_depth`: depth of each tree. Kept small (3–5) because GB corrects errors sequentially — shallow trees suffice.
- `subsample`: fraction of rows used per tree (< 1.0 adds randomness, like RF's bootstrap). Called *stochastic gradient boosting*.

**Why it can beat random forest:**
RF builds trees independently and averages. GB builds each tree to fix what the previous ensemble got wrong — more directed, so it typically needs fewer trees to reach the same accuracy.

**Benchmark to beat:** RF tuned CV RMSE = **$122.60**

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_predict, RandomizedSearchCV

%matplotlib inline

In [2]:
_cwd = Path().resolve()
ROOT = _cwd if (_cwd / "data").exists() else _cwd.parent
DATA = ROOT / "data"

train = pd.read_csv(DATA / "train.csv")
test  = pd.read_csv(DATA / "test.csv")

train = train[
    (train['price'] > 0) & (train['airline'] != 'Southern_Airways_Express')
].reset_index(drop=True)

KF = KFold(n_splits=5, shuffle=True, random_state=42)

STOPS_MAP = {'zero': 0, 'one': 1, 'two_or_more': 2}
TIME_MAP  = {'Early_Morning': 0, 'Morning': 1, 'Afternoon': 2,
             'Evening': 3, 'Night': 4, 'Late_Night': 5}
CLASS_MAP = {'Economy': 0, 'Business': 1}

train_enc = train.assign(
    stops          = train['stops'].map(STOPS_MAP),
    departure_time = train['departure_time'].map(TIME_MAP),
    arrival_time   = train['arrival_time'].map(TIME_MAP),
    flight_class   = train['flight_class'].map(CLASS_MAP),
).drop(columns=['id', 'flight'])

X = train_enc.drop(columns=['price'])
y = train['price'].to_numpy()

print(f"train: {train.shape}   X: {X.shape}")

train: (18050, 12)   X: (18050, 9)


In [3]:
NUM_COLS = ['duration', 'days_left', 'stops', 'departure_time', 'arrival_time', 'flight_class']
OHE_COLS = ['airline', 'source_city', 'destination_city']

def make_preprocessor():
    # GB is scale-invariant; passthrough numeric/ordinal, OHE categoricals
    return ColumnTransformer([
        ('pass', 'passthrough', NUM_COLS),
        ('ohe',  OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), OHE_COLS),
    ])

def make_pipeline(model):
    return Pipeline([('pre', make_preprocessor()), ('model', model)])

def cv_rmse(model, X, y, log_target=False):
    y_fit = np.log1p(y) if log_target else y
    preds = cross_val_predict(model, X, y_fit, cv=KF)
    if log_target:
        preds = np.expm1(preds)
    return float(np.sqrt(np.mean((y - preds) ** 2)))

In [4]:
# GB baseline — sklearn defaults
# Note: GradientBoostingRegressor is single-threaded (no n_jobs).
# This cell takes ~2 min. Tuning below takes ~15 min.
pipe_base = make_pipeline(GradientBoostingRegressor(random_state=42))
rmse_base = cv_rmse(pipe_base, X, y)
print(f"GB  baseline CV RMSE : ${rmse_base:.2f}")
print(f"RF  tuned    CV RMSE : $122.60")

GB  baseline CV RMSE : $139.51
RF  tuned    CV RMSE : $122.60


In [5]:
# Hyperparameter tuning
# learning_rate and n_estimators trade off: lower lr needs more trees.
# max_depth kept shallow (3–5) as is standard for boosting.
# subsample < 1.0 adds stochastic noise — often helps generalisation.
param_dist = {
    'model__n_estimators'    : [100, 200, 300],
    'model__learning_rate'   : [0.05, 0.1, 0.2],
    'model__max_depth'       : [3, 4, 5],
    'model__subsample'       : [0.8, 1.0],
    'model__min_samples_leaf': [5, 10, 20],
}

rs = RandomizedSearchCV(
    make_pipeline(GradientBoostingRegressor(random_state=42)),
    param_distributions=param_dist,
    n_iter=15,
    cv=KF,
    scoring='neg_root_mean_squared_error',
    random_state=42,
)
rs.fit(X, y)

print("Best params:")
for k, v in rs.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nBest CV RMSE (tuning score): ${-rs.best_score_:.2f}")

Best params:
  model__subsample: 0.8
  model__n_estimators: 300
  model__min_samples_leaf: 10
  model__max_depth: 5
  model__learning_rate: 0.1

Best CV RMSE (tuning score): $123.50


In [6]:
# Best GB — final CV RMSE + feature importance
best_params = {k.replace('model__', ''): v for k, v in rs.best_params_.items()}
pipe_best = make_pipeline(GradientBoostingRegressor(**best_params, random_state=42))

rmse_best = cv_rmse(pipe_best, X, y)
print(f"GB  tuned CV RMSE : ${rmse_best:.2f}")
print(f"RF  tuned CV RMSE : $122.60   delta: {122.60 - rmse_best:+.2f}")

pipe_best.fit(X, y)

feature_names = (
    NUM_COLS +
    list(pipe_best.named_steps['pre']
                  .named_transformers_['ohe']
                  .get_feature_names_out(OHE_COLS))
)
importances = pipe_best.named_steps['model'].feature_importances_

top_n = min(20, len(importances))
idx   = np.argsort(importances)[-top_n:]

fig, ax = plt.subplots(figsize=(8, 6), layout='constrained')
ax.barh(range(top_n), importances[idx], edgecolor='black')
ax.set_yticks(range(top_n))
ax.set_yticklabels([feature_names[i] for i in idx])
ax.set_xlabel('Feature importance')
ax.set_title(f'Top {top_n} features — best GB')
plt.show()

GB  tuned CV RMSE : $126.54
RF  tuned CV RMSE : $122.60   delta: -3.94


RecursionError: maximum recursion depth exceeded

Error in callback <function _draw_all_if_interactive at 0x73d9f5848bf0> (for post_execute), with arguments args (),kwargs {}:


RecursionError: maximum recursion depth exceeded

RecursionError: maximum recursion depth exceeded

<Figure size 800x600 with 1 Axes>

In [ ]:
# Save best pipeline
import pickle

out_path = ROOT / "outputs" / "model_gb_v1.pkl"
with open(out_path, "wb") as f:
    pickle.dump(pipe_best, f)
print(f"Saved pipeline to {out_path}")

In [ ]:
# Comparison table so far
results = [
    {'model': 'M1  LinearRegression  numeric only',    'CV RMSE ($)': 207.00},
    {'model': 'M5a Lasso',                             'CV RMSE ($)': 170.25},
    {'model': 'M5b ElasticNet',                        'CV RMSE ($)': 167.80},
    {'model': 'M4  Ridge',                             'CV RMSE ($)': 167.02},
    {'model': 'M3  LinearRegression  log + last_week', 'CV RMSE ($)': 167.01},
    {'model': 'M2  LinearRegression  all features',    'CV RMSE ($)': 166.75},
    {'model': 'RF  baseline',                          'CV RMSE ($)': 129.03},
    {'model': 'RF  tuned',                             'CV RMSE ($)': 122.60},
    {'model': 'GB  baseline',                          'CV RMSE ($)': round(rmse_base, 2)},
    {'model': f'GB  tuned',                            'CV RMSE ($)': round(rmse_best, 2)},
]

summary = (pd.DataFrame(results)
             .sort_values('CV RMSE ($)')
             .reset_index(drop=True))
summary.index += 1
print(summary.to_string())